### Define variables
Initial gb and fasta files from entire project should be first defined.

In [53]:
workdir = "plastomes/table2asn/cp"
cc_gb = "plastomes/table2asn/cc/fixed_table2asn/output.gbf"
cc_fna = "plastomes/table2asn/fasta_manual_description/Crepis_callicephala.fasta"

cp_gb = "fixed_Crepis_purpurea.gb"
cp_fna = "plastomes/final/gb2sequin_out/c_purpurea_final/03/Crepis_purpurea.fsa"

### Check CDS of genes with known RNA editing sites

In [3]:
from Bio import SeqIO, SeqRecord

# output psbL data
gb_file = cp_gb # file with fixed tRNAs
target_gene = "psbL"

with open(gb_file) as handle:
    for record in SeqIO.parse(handle, "gb"):
        for feature in record.features:
            if feature.type in ["CDS", "gene"]:
                if feature.qualifiers.get('gene')[0] == target_gene:
                    print(feature)
                    cds_seq = feature.extract(record.seq)
                    #print(feature.qualifiers.get('gene')[0], cds_seq[:20])
                    start = feature.location.start - 1
                    end = start + 3
                    codon = record.seq[start:end].reverse_complement()  # 0-based
                    #print(f"codon: {codon}")

type: gene
location: [61908:62025](-)
qualifiers:
    Key: gene, Value: ['psbL']

type: CDS
location: [61908:62025](-)
qualifiers:
    Key: gene, Value: ['psbL']
    Key: product, Value: ['photosystem II subunit L']
    Key: transl_table, Value: ['11']
    Key: translation, Value: ['TTQSNPNEQNVELNRTSLYWGLLLIFVLAVLFSNYFFN']



The _**psbL**_ has no canonical codon in coding sequence start position.
Exception should be added and transcription output should be modified by changing the first AA letter to M

In [4]:
# output ndhD data
target_gene = "ndhD"

aa_shift = 6
bp_shift = aa_shift * 3

for record in SeqIO.parse(gb_file, "gb"):
    for feature in record.features:
        if feature.type == "CDS":
            gene_name = feature.qualifiers.get("gene", [""])[0]
            if gene_name == target_gene:
                print(feature)
                
                start = int(feature.location.start)
                end = int(feature.location.end)
                strand = feature.location.strand
                na = record.seq[start : end]

                if strand == 1:
                    position_shifted = start + bp_shift
                    na_shifted = record.seq[start + bp_shift : end]
                    ncbi_coord_str = f"{position_shifted + 1}..{end}"
                elif strand == -1:
                    position_shifted = end - bp_shift
                    na_shifted = record.seq[start : end - bp_shift]
                    ncbi_coord_str = f"{start + 1}..{position_shifted}"
                else:
                    print("Strand orientation unknown.")
                    continue
                
                for na_seq in [na, na_shifted]:
                    # convert to 5'-3' before translation
                    coding_seq = na_seq.reverse_complement() if strand == -1 else na_seq
                    aa_seq = coding_seq.translate(to_stop=False)
                    if na_seq == na_shifted:
                        print(f"Translated sequence first symbols (shifted {aa_shift} aa): {aa_seq[0:15]}")
                    else:
                        print(f"Translated sequence first symbols: {aa_seq[0:21]}")
                    

                print(f"\ngene strand: ({'+' if strand == 1 else '-'})")
                print(f"Original gene coordinates in Biopython are [{start}:{end}]")
                print(f"Original GenBank coordinates: {start + 1}..{end}")
                print(f"Adjusted (start for (+) or end for (-) strand) position after shifting by {aa_shift} AA in Biopython is {position_shifted}")
                print(f"New GenBank coordinates after {aa_shift} aa shift: {ncbi_coord_str}")

type: CDS
location: [112421:113942](-)
qualifiers:
    Key: gene, Value: ['ndhD']
    Key: product, Value: ['NADH dehydrogenase subunit D']
    Key: transl_table, Value: ['11']
    Key: translation, Value: ['VYLVFTTNKFPWLTIIVVLPIFAGSFILFFPHKGNRVIRWYTICICMLELILTTYAFCYHFQLDDPLIQLVEDYKWINFFDFRWKLGIDGLSIGPVLLTGFITTLATLAAWPVTRDSRLFHFLMLAMYSGQIGSFSSRDLLLFFIMWELELIPVYLLLSMWGGKKRLYSATKFILYTAGGSIFLLMGVLGVGLYGSNEPTLNFETSVNQSYPVALEIIFYIGFFIAFAVKSPILPLHTWLPDTHGEAHYSTCMLLAGILLKMGAYGLIRINMELLPHAHSIFAPWLMIVGTIQIIYAASTSPGQRNLKKRIAYSSVSHMGFILIGIASITDTGLNGAILQIISHGFIGAALFFLAGTSYDRIRLVYLDQMGGVAIPMPKIFTMFSSFSMASLALPGMSGFVAEVMVFLGIITSQKYLLIPKIAITFVMAIGMILTPIYSLSMLRQMFYGYKLFNTPNSYVFDSGPRELFVSISIFLPVIGIGMYPDFVLSLSVDKVEGILSNYFYR']

Translated sequence first symbols: VYLVFTTNKFPWLTIIVVLPI
Translated sequence first symbols (shifted 6 aa): TNKFPWLTIIVVLPI

gene strand: (-)
Original gene coordinates in Biopython are [112421:113942]
Original GenBank coordinates: 112422..113942
Adjusted (start for (+) or end for (-

As we see, in Crepis purpurea, the _**ndhD**_ has 6 additional amino acids in the start of its CDS sequence, but C. callicephala has not. It may represent the annotation error in C. purpurea and should be fixed by adjusting the gene boundaries.

It also has non-canonical codon at the coding sequence start position. The first AA should be changed to M after adding exception.

So there are two issues:
1. Start position needs adjustment.
2. CDS translation needs modifying by changing the first AA letter to M.

### Fixing non-canonical start codon-related issues via code
#### Granular approach
As we need changing two genes with different combination of things to change, the most preferred is a granular step-by-step approach, when we:
1. Adjust _**ndhD**_ coordinates in `CDS` and `gene` features.
2. Fix start position issues in `CDS` features of _**ndhD**_ and _**psbL**_.

In [5]:
# 1. Adjust coordinates
from Bio import SeqIO
from Bio.SeqFeature import FeatureLocation
import os

# Configuration
input_gb = cp_gb
outdir = f"{workdir}/01_prot_ndhD"
output_gb = f"{outdir}/cp_ndhD-coord-fixed.gb"
target_gene = "ndhD"
aa_shift = 6
bp_shift = aa_shift * 3

os.makedirs(outdir, exist_ok=True)

# Load all records into memory
records = list(SeqIO.parse(input_gb, "gb"))

for record in records:
    for feature in record.features:
        # Safely check for target gene
        gene_name = feature.qualifiers.get("gene", [""])[0]
        if feature.type in ["gene", "CDS"] and gene_name == target_gene:
            old_start = int(feature.location.start)
            old_end = int(feature.location.end)
            strand = int(feature.location.strand)

            # Calculate new boundaries
            if strand == -1:
                # Reverse strand: downstream shift reduces the end coordinate
                new_start = old_start
                new_end = old_end - bp_shift
            elif strand == 1:
                # Forward strand: downstream shift increases the start coordinate
                new_start = old_start + bp_shift
                new_end = old_end
            else:
                continue

            # Validate that new length maintains open reading frame
            if (new_end - new_start) % 3 != 0:
                raise ValueError(f"New length for {target_gene} is not a multiple of 3.")

            # Update feature location (0-based, half-open)
            feature.location = FeatureLocation(new_start, new_end, strand)
            print(f"[{feature.type}] feature coords of {gene_name} for target gene {target_gene} was updated: {old_start}..{old_end} -> {new_start}..{new_end} (0-based)")

            # update feature translation length
            if feature.type == 'CDS':
                trans_old = feature.qualifiers.get('translation', "")[0]
                # convert to 5'-3' before translation
                coding_seq = na_seq.reverse_complement() if strand == -1 else na_seq
                trans_new = coding_seq.translate(to_stop=True)
                feature.qualifiers['translation'] = [trans_new]
                print(f"old translation: {trans_old}\nnew translation: {trans_new}")

# Write corrected file
SeqIO.write(records, output_gb, "gb")
print(f"\nSaved corrected GenBank file to: {output_gb}")

[gene] feature coords of ndhD for target gene ndhD was updated: 112421..113942 -> 112421..113924 (0-based)
[CDS] feature coords of ndhD for target gene ndhD was updated: 112421..113942 -> 112421..113924 (0-based)
old translation: VYLVFTTNKFPWLTIIVVLPIFAGSFILFFPHKGNRVIRWYTICICMLELILTTYAFCYHFQLDDPLIQLVEDYKWINFFDFRWKLGIDGLSIGPVLLTGFITTLATLAAWPVTRDSRLFHFLMLAMYSGQIGSFSSRDLLLFFIMWELELIPVYLLLSMWGGKKRLYSATKFILYTAGGSIFLLMGVLGVGLYGSNEPTLNFETSVNQSYPVALEIIFYIGFFIAFAVKSPILPLHTWLPDTHGEAHYSTCMLLAGILLKMGAYGLIRINMELLPHAHSIFAPWLMIVGTIQIIYAASTSPGQRNLKKRIAYSSVSHMGFILIGIASITDTGLNGAILQIISHGFIGAALFFLAGTSYDRIRLVYLDQMGGVAIPMPKIFTMFSSFSMASLALPGMSGFVAEVMVFLGIITSQKYLLIPKIAITFVMAIGMILTPIYSLSMLRQMFYGYKLFNTPNSYVFDSGPRELFVSISIFLPVIGIGMYPDFVLSLSVDKVEGILSNYFYR
new translation: TNKFPWLTIIVVLPIFAGSFILFFPHKGNRVIRWYTICICMLELILTTYAFCYHFQLDDPLIQLVEDYKWINFFDFRWKLGIDGLSIGPVLLTGFITTLATLAAWPVTRDSRLFHFLMLAMYSGQIGSFSSRDLLLFFIMWELELIPVYLLLSMWGGKKRLYSATKFILYTAGGSIFLLMGVLGVGLYGSNEPTLNFETSVNQSYPVALEIIFYIGFFIAFAVKSPILPLHTWLPDTHGEAHYSTC

In [6]:
# check gene output
target_gene = "ndhD"

aa_shift = 6
bp_shift = aa_shift * 3
gb_old = cp_gb
gb_new = output_gb

for gb_file in [gb_old, gb_new]:
    for record in SeqIO.parse(gb_file, "gb"):
        print(f"______________________________________________\nGenBank file: {gb_file}")
        for feature in record.features:
            if feature.type == "CDS":
                gene_name = feature.qualifiers.get("gene", [""])[0]
                if gene_name == target_gene:
                    #print(feature)
                    
                    start = int(feature.location.start)
                    end = int(feature.location.end)
                    strand = feature.location.strand
                    na = record.seq[start : end]

                    if strand == 1:
                        position_shifted = start + bp_shift
                        na_shifted = record.seq[start + bp_shift : end]
                        ncbi_coord_str = f"{position_shifted + 1}..{end}"
                    elif strand == -1:
                        position_shifted = end - bp_shift
                        na_shifted = record.seq[start : end - bp_shift]
                        ncbi_coord_str = f"{start + 1}..{position_shifted}"
                    else:
                        print("Strand orientation unknown.")
                        continue
                    
                    for na_seq in [na]:
                        # convert to 5'-3' before translation
                        coding_seq = na_seq.reverse_complement() if strand == -1 else na_seq
                        aa_seq = coding_seq.translate(to_stop=False)
                        print(f"Translated sequence first symbols: {aa_seq[0:21]}")
                        

                    print(f"\ngene strand: ({'+' if strand == 1 else '-'})")
                    print(f"Original coordinates in Biopython: [{start}:{end}]")
                    print(f"Original coordinates in GenBank: {start + 1}..{end}")
                    print(f"CDS feature translation: {feature.qualifiers.get('translation', "")}")

______________________________________________
GenBank file: fixed_Crepis_purpurea.gb
Translated sequence first symbols: VYLVFTTNKFPWLTIIVVLPI

gene strand: (-)
Original coordinates in Biopython: [112421:113942]
Original coordinates in GenBank: 112422..113942
CDS feature translation: ['VYLVFTTNKFPWLTIIVVLPIFAGSFILFFPHKGNRVIRWYTICICMLELILTTYAFCYHFQLDDPLIQLVEDYKWINFFDFRWKLGIDGLSIGPVLLTGFITTLATLAAWPVTRDSRLFHFLMLAMYSGQIGSFSSRDLLLFFIMWELELIPVYLLLSMWGGKKRLYSATKFILYTAGGSIFLLMGVLGVGLYGSNEPTLNFETSVNQSYPVALEIIFYIGFFIAFAVKSPILPLHTWLPDTHGEAHYSTCMLLAGILLKMGAYGLIRINMELLPHAHSIFAPWLMIVGTIQIIYAASTSPGQRNLKKRIAYSSVSHMGFILIGIASITDTGLNGAILQIISHGFIGAALFFLAGTSYDRIRLVYLDQMGGVAIPMPKIFTMFSSFSMASLALPGMSGFVAEVMVFLGIITSQKYLLIPKIAITFVMAIGMILTPIYSLSMLRQMFYGYKLFNTPNSYVFDSGPRELFVSISIFLPVIGIGMYPDFVLSLSVDKVEGILSNYFYR']
______________________________________________
GenBank file: plastomes/table2asn/cp/01_prot_ndhD/cp_ndhD-coord-fixed.gb
Translated sequence first symbols: TNKFPWLTIIVVLPIFAGSFI

gene strand: (-)
Original 

In [7]:
from Bio import SeqIO

input_gb = "plastomes/table2asn/cp/01_prot_ndhD/cp_ndhD-coord-fixed.gb"      # Define file directly by relative path
outdir=f"{workdir}/02_start_codon"
output_gb = f"{outdir}/cp_start_codons_fixed.gb"
target_genes = {"psbL", "ndhD"}

os.makedirs(outdir, exist_ok=True)

records = list(SeqIO.parse(input_gb, "gb"))

for record in records:
    for feature in record.features:
        if feature.type == "CDS" and feature.qualifiers.get("gene", [""])[0] in target_genes:
            gene = feature.qualifiers["gene"][0]
            loc = feature.location
            
            # 1. Extract genomic start codon & determine 1-based coordinates
            if loc.strand == -1:
                # Reverse strand: start codon is at the 3' end of the genomic interval
                genomic_codon = record.seq[loc.end-3:loc.end].reverse_complement()
                start_codon_pos = f"complement({loc.end-2}..{loc.end})"
            else:
                # Forward strand: start codon is at the 5' end
                genomic_codon = record.seq[loc.start:loc.start+3]
                start_codon_pos = f"{loc.start+1}..{loc.start+3}"
                
            genomic_codon_str = str(genomic_codon).upper()
            
            # Skip if already standard ATG
            if genomic_codon_str == "ATG":
                print(f"{gene}: Already has standard ATG start. No change needed.")
                continue
                
            # 2. Update /translation to start with M (edited protein)
            if "translation" in feature.qualifiers:
                current_trans = feature.qualifiers["translation"][0]
                if current_trans[0] != "M":
                    feature.qualifiers["translation"][0] = "M" + current_trans[1:]
                    print(f"{gene}: Updated translation to start with M.")
                    
            # 3. Add /transl_except qualifier (NCBI official format)
            except_val = f"(pos:{start_codon_pos},aa=M)"
            feature.qualifiers["transl_except"] = [except_val]
            print(f"{gene}: Added /transl_except=\"{except_val}\"")
            
            # 4. Add clarifying note (optional but recommended for reviewers)
            if "note" not in feature.qualifiers:
                feature.qualifiers["note"] = []
            feature.qualifiers["note"].append("Start codon created by RNA editing (C-to-U)")

SeqIO.write(records, output_gb, "gb")
print(f"\nSaved corrected GenBank to: {output_gb}")

psbL: Updated translation to start with M.
psbL: Added /transl_except="(pos:complement(62023..62025),aa=M)"
ndhD: Updated translation to start with M.
ndhD: Added /transl_except="(pos:complement(113922..113924),aa=M)"

Saved corrected GenBank to: plastomes/table2asn/cp/02_start_codon/cp_start_codons_fixed.gb


In [8]:
# Check CDS for start codon exception
from Bio import SeqIO

gb_file = "plastomes/table2asn/cp/02_start_codon/cp_start_codons_fixed.gb" # file with fixed tRNAs
genes = ["psbL", "ndhD"]

with open(gb_file) as handle:
    for record in SeqIO.parse(handle, "gb"):
        for feature in record.features:
            if feature.qualifiers.get('gene', "[]")[0] in genes:
                if feature.type in ["CDS"]:
                    print(feature)

type: CDS
location: [61908:62025](-)
qualifiers:
    Key: gene, Value: ['psbL']
    Key: note, Value: ['Start codon created by RNA editing (C-to-U)']
    Key: product, Value: ['photosystem II subunit L']
    Key: transl_except, Value: ['(pos:complement(62023..62025),aa=M)']
    Key: transl_table, Value: ['11']
    Key: translation, Value: ['MTQSNPNEQNVELNRTSLYWGLLLIFVLAVLFSNYFFN']

type: CDS
location: [112421:113924](-)
qualifiers:
    Key: gene, Value: ['ndhD']
    Key: note, Value: ['Start codon created by RNA editing (C-to-U)']
    Key: product, Value: ['NADH dehydrogenase subunit D']
    Key: transl_except, Value: ['(pos:complement(113922..113924),aa=M)']
    Key: transl_table, Value: ['11']
    Key: translation, Value: ['MNKFPWLTIIVVLPIFAGSFILFFPHKGNRVIRWYTICICMLELILTTYAFCYHFQLDDPLIQLVEDYKWINFFDFRWKLGIDGLSIGPVLLTGFITTLATLAAWPVTRDSRLFHFLMLAMYSGQIGSFSSRDLLLFFIMWELELIPVYLLLSMWGGKKRLYSATKFILYTAGGSIFLLMGVLGVGLYGSNEPTLNFETSVNQSYPVALEIIFYIGFFIAFAVKSPILPLHTWLPDTHGEAHYSTCMLLAGILLKMGAYGLIRI

### Compare features between cc and cp annotations
Prepare pandas dataframe that harbours all feature data

In [9]:
import pandas as pd
import numpy as np
from Bio import SeqRecord, SeqIO, SeqFeature

# variables
c_pur = "plastomes/table2asn/cp/02_start_codon/cp_start_codons_fixed.gb"
c_cal = "plastomes/table2asn/cc/fixed_table2asn/output.gbf"

regions = {}
dataset = []

# Do initial parsing
for acc in c_cal, c_pur:
    with open(acc, "r") as handle:
        for record in SeqIO.parse(handle, "genbank"):
            features = record.features
            # check if there features other than 'source'
            if len(features) > 1:
                for feature in features:
                    # check for genome regions
                    if feature.type in ['misc_feature', 'repeat_region']:
                        label = feature.qualifiers.get('note', "")[0].split()[-1].strip("()")
                        regions[label] = [int(feature.location.start), int(feature.location.end)]
                # filling the dataframe
                if acc == c_cal:
                    organism = "Crepis callicephala"
                else:
                    organism = record.annotations.get('organism', "")
                for feature in features:
                    feature_type = feature.type
                    if feature_type in ['gene', 'CDS', 'tRNA', 'rRNA']:
                        feature_dict = {}
                        # gene and start position need for further check
                        gene = feature.qualifiers.get('gene', "")[0]
                        start = int(feature.location.start)
                        end = int(feature.location.end)
                        if feature_type == 'gene':
                            feature_dict['organism'] = organism
                            feature_dict['gene'] = gene
                            feature_dict['start'] = start
                            feature_dict['end'] = end
                            feature_dict['gene_len'] = int(len(feature))
                            feature_dict['spanned'] = False
                            if feature.location.strand:
                                feature_dict['strand'] = int(feature.location.strand)

                            for r_name, r_range in regions.items():
                                if r_range[0] <= start < r_range[1]:
                                    feature_dict['region'] = r_name
                                    feature_dict['spanned'] = (end >= r_range[1])
                                    break

                            dataset.append(feature_dict)

                        if feature_type in ['CDS', 'tRNA', 'rRNA']:
                            for feat in dataset[-4:]:
                                if gene == feat['gene'] and start == feat['start']:
                                    feat['type'] = feature_type
                                    feat['feature_len'] = len(feature)
                                    sequence = feature.extract(record.seq)
                                    feat['na_seq'] = f"{sequence[:6]}...{sequence[-6:]}"
                                    if feature_type == 'CDS':
                                        aa_seq = feature.qualifiers.get('translation', [""])[0]
                                        feat['aa_seq'] = aa_seq[:6]
                                    break  # dataset is already updated here
                                    

df = pd.DataFrame(dataset)
#print(df.head())
display(df)

,organism,gene,start,end,gene_len,spanned,strand,region,type,feature_len,na_seq,aa_seq
0,Crepis callicephala,rps12,67368,95793,907,True,-1.0,LSC,CDS,372.0,ATGCCA...AAATAA,MPTIKQ
1,Crepis callicephala,trnH-GUG,2,77,75,False,-1.0,LSC,tRNA,75.0,GCGGAT...TCGCCC,NaN
2,Crepis callicephala,psbA,474,1536,1062,False,-1.0,LSC,CDS,1062.0,ATGACT...GGATAA,MTAILE
3,Crepis callicephala,trnK-UUU,1758,4358,2600,False,-1.0,LSC,tRNA,74.0,TGGGTT...ACCCAT,NaN
4,Crepis callicephala,matK,2079,3591,1512,False,-1.0,LSC,CDS,1512.0,ATGGAG...GAATGA,MEKFQS
...,...,...,...,...,...,...,...,...,...,...,...,...
266,Crepis purpurea,ycf2,140899,147754,6855,False,-1.0,IRA,CDS,6855.0,ATGACA...CCATGA,MTGHEF
267,Crepis purpurea,trnM-CAU,147865,147940,75,False,1.0,IRA,tRNA,75.0,GCATCC...ATGCAC,NaN
268,Crepis purpurea,rpl23,148104,148386,282,False,1.0,IRA,CDS,282.0,ATGGAT...ACTTAA,MDGIRY
269,Crepis purpurea,rpl2,148404,149894,1490,False,1.0,IRA,CDS,825.0,ATGGCG...AAATAG,MAIHLY


In [10]:
df_CDS = df[df['type'] == 'CDS']
df_CDS 
#display(df_CDS)

pivot_CDS_len = df_CDS.pivot_table(
    values='feature_len',
    index='gene',
    columns=['organism'],
    aggfunc='first'
)

piv = pivot_CDS_len.sort_index(axis=1)

# Keep only rows where values differ (excluding NaN mismatches)
diff_mask = (piv['Crepis callicephala'] != piv['Crepis purpurea']) #& \
#            piv['Crepis callicephala'].notna() & \
#            piv['Crepis purpurea'].notna()

diff_genes = piv[diff_mask]
display(diff_genes)

organism,Crepis callicephala,Crepis purpurea
gene,,
petD,525.0,483.0
psaA,2253.0,1776.0
rpl22,465.0,540.0
rpoC1,2076.0,1791.0
rpoC2,4125.0,2274.0


### Investigate start codones among annotations



In [11]:
pivot_CDS_aa = df_CDS.pivot_table(
    values='aa_seq',
    index='gene',
    columns=['organism'],
    aggfunc='first'
)

piv = pivot_CDS_aa.sort_index(axis=1)

# Keep only rows where values differ (excluding NaN mismatches)
diff_mask = (piv['Crepis callicephala'] != piv['Crepis purpurea']) #& \
#            piv['Crepis callicephala'].notna() & \
#            piv['Crepis purpurea'].notna()

diff_genes = piv[diff_mask]
display(diff_genes)

organism,Crepis callicephala,Crepis purpurea
gene,,
petD,MSGSYG,MGVTKK
rps19,MTRSLK,VTRSLK


In [12]:
pivot_CDS_na = df_CDS.pivot_table(
    values='na_seq',
    index='gene',
    columns=['organism'],
    aggfunc='first'
)

piv = pivot_CDS_na.sort_index(axis=1)

# Keep only rows where values differ (excluding NaN mismatches)
diff_mask = (piv['Crepis callicephala'] != piv['Crepis purpurea']) #& \
#            piv['Crepis callicephala'].notna() & \
#            piv['Crepis purpurea'].notna()

diff_genes = piv[diff_mask]
display(diff_genes)

organism,Crepis callicephala,Crepis purpurea
gene,,
petD,ATGTCC...TTTTAA,ATGGGA...TTTTAA
psaA,ATGATT...GGATAA,ATGATT...TCTTAG
rpl22,ATGTTA...AAATAA,ATGTTA...TTCTGA
rpoC1,ATGATC...GGTTAA,ATGATC...ATTTAG
rpoC2,ATGGCA...TTTTAG,ATGGCA...AATTGA


In [13]:
issued_genes = ['rps19', 'petD', 'psaA', 'rpl22', 'rpoC1', 'rpoC2']

df_lookup = df[df['gene'].isin(issued_genes)].copy()
# Pivot with multiple value columns
pivot_multi = df_lookup.pivot_table(
    index='gene',
    columns='organism',
    values=['start', 'end', 'feature_len', 'na_seq', 'aa_seq', 'strand'],
    aggfunc='first'  # or 'first' since each gene/organism combo should be unique
)

# Optional: Flatten column MultiIndex for easier reading
pivot_multi.columns = ['_'.join(col).strip() for col in pivot_multi.columns.values]

display(pivot_multi.head())

,aa_seq_Crepis callicephala,aa_seq_Crepis purpurea,end_Crepis callicephala,end_Crepis purpurea,feature_len_Crepis callicephala,feature_len_Crepis purpurea,na_seq_Crepis callicephala,na_seq_Crepis purpurea,start_Crepis callicephala,start_Crepis purpurea,strand_Crepis callicephala,strand_Crepis purpurea
gene,,,,,,,,,,,,
petD,MSGSYG,MGVTKK,75361,75387,525.0,483.0,ATGTCC...TTTTAA,ATGGGA...TTTTAA,74836,74204,1.0,1.0
psaA,MIIRSP,MIIRSP,41258,41293,2253.0,1776.0,ATGATT...GGATAA,ATGATT...TCTTAG,39005,39517,-1.0,-1.0
rpl22,MLNKRT,MLNKRT,81529,81543,465.0,540.0,ATGTTA...AAATAA,ATGTTA...TTCTGA,81064,81003,-1.0,-1.0
rpoC1,MIDRYT,MIDRYT,18963,18702,2076.0,1791.0,ATGATC...GGTTAA,ATGATC...ATTTAG,16164,16182,1.0,1.0
rpoC2,MAERPT,MAERPT,23201,21367,4125.0,2274.0,ATGGCA...TTTTAG,ATGGCA...AATTGA,19076,19093,1.0,1.0


In [14]:
# comparing translations, first cc, then cp

for issued in issued_genes:
    print(f"\n{issued}")
    for gb in c_cal, c_pur:
        records = SeqIO.parse(gb, "gb")
        for record in records:
            for feature in record.features:
                if feature.type == 'CDS':
                    gene = feature.qualifiers['gene'][0]
                    if gene == issued:
                        cds = feature.extract(record.seq)
                        aa = cds.translate(to_stop=False)
                        print(f"{aa}")


rps19
VTRSLKKNPFVANNLLKKINKLNTKAEKEIIITWSRASTIIPIMVGHTIAIHNGKEHLPIYITDRMVGHKLGEFAPTLNFRGHAKSDNRSRR*
VTRSLKKNPFVANNLLKKINKLNTKAEKEIIITWSRASTIIPIMVGHTIAIHNGKEHLPIYITDRMVGHKLGEFAPTLNFRGHAKSDNRSRR*

petD
MSGSYGGWIYKNSPIPITKKPDLNDPVLRAKLAKGMGHNYYGEPAWPNDLLYIFPVVILGTIACNVGLAVLEPSMIGEPADPFATPLEILPEWYFFPVFQILRTVPNKLLGVLLMVSVPAGLLTVPFLENVNKFQNPFRRPVATTVFLIGTAVALWLGIGATLPIDKSLTLGLF*
MGVTKKPDLNDPVLRAKLAKGMGHNYYGEPAWPNDLLYIFPVVILGTIACNVGLAVLEPSMIGEPADPFATPLEILPEWYFFPVFQILRTVPNKLLGVLLMVSVPAGLLTVPFLENVNKFQNPFRRPVATTVFLIGTAVALWLGIGATLPIDKSLTLGLF*

psaA
MIIRSPEPEVKILVDRDHIKTSFEEWARPGHFSRTIAKGPETTTWIWNLHADAHDFDSHTSDLEEISRKVFSAHFGQLSIIFLWLSGMYFHGARFSNYEAWLSDPTHIGPSAQVVWPIVGQEILNGDVGGGFRGIQITSGFFQIWRASGITSELQLYCTAIGALVFAALMLFAGWFHYHKAAPKLAWFQDVESMLNHHLAGLLGLGSLSWAGHQVHVSLPINQFLNAGVDPKEIPLPHEFILNRDLLAQLYPSFAEGATPFFTLNWSKYADFLTFRGGLDPVTGGLWLTDTAHHHLAIAILFLIAGHMYRTNWGIGHGLKDILEAHKGPFTGQGHKGLYEILTTSWHAQLSLNLAMLGSLTIVVAHHMYAMPPYPYLATDYGTVLSLFTHHMWIGGFLIVGAAAHAAIFMVRDYDPTTRYNDLLDRVLRHRDAIISHLNWACIFLGFHSFGLYIH

Save reference sequences for issued genes

In [16]:
import os
import time
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

Entrez.email = "asantaraqtasli@gmail.com"

accessions = {
    "NC_007578.1": "Lactuca sativa",
    "NC_016893.1": "Lapsana communis",
    "NC_046516.1": "Youngia japonica",
    "NC_031396.1": "Taraxacum mongolicum",
    "NC_061360.1": "Crepis rigescens",
}
out_dir = "plastomes/refs"
os.makedirs(out_dir, exist_ok=True)

for acc, species in accessions.items():
    tag = species.lower().replace(" ", "_")
    print(f"Fetching {species} ({acc})...")
    
    with Entrez.efetch(db="nucleotide", id=acc, rettype="gb", retmode="text") as handle:
        rec = SeqIO.read(handle, "gb")
        
    for gene in issued_genes:
        cds_feat, gene_feat = None, None
        for feat in rec.features:
            g = feat.qualifiers.get("gene", [""])[0]
            if g == gene:
                if feat.type == "CDS":
                    cds_feat = feat
                elif feat.type == "gene":
                    gene_feat = feat
                    
        if not cds_feat:
            print(f"  Skipping {gene}: CDS not found.")
            continue
            
        cds_seq = cds_feat.extract(rec.seq)
        genomic_seq = gene_feat.extract(rec.seq) if gene_feat else cds_feat.extract(rec.seq)
        prot_seq = Seq(cds_feat.qualifiers.get("translation", [str(cds_seq.translate())])[0])
        
        prefix = f"{gene}_{tag}"
        seq_map = {
            "genomic.fna": genomic_seq,
            "cds.fna": cds_seq,
            "protein.faa": prot_seq
        }
        
        for ext, seq in seq_map.items():
            filepath = os.path.join(out_dir, f"{prefix}.{ext}")
            SeqIO.write(SeqRecord(seq, id=prefix), filepath, "fasta")
            
        print(f"  Saved {prefix}.*")
        time.sleep(0.5)

Fetching Lactuca sativa (NC_007578.1)...
  Saved rps19_lactuca_sativa.*
  Saved petD_lactuca_sativa.*
  Saved psaA_lactuca_sativa.*
  Saved rpl22_lactuca_sativa.*
  Saved rpoC1_lactuca_sativa.*
  Saved rpoC2_lactuca_sativa.*
Fetching Lapsana communis (NC_016893.1)...
  Skipping rps19: CDS not found.
  Skipping petD: CDS not found.
  Skipping psaA: CDS not found.
  Skipping rpl22: CDS not found.
  Skipping rpoC1: CDS not found.
  Skipping rpoC2: CDS not found.
Fetching Youngia japonica (NC_046516.1)...
  Saved rps19_youngia_japonica.*
  Saved petD_youngia_japonica.*
  Saved psaA_youngia_japonica.*
  Saved rpl22_youngia_japonica.*
  Saved rpoC1_youngia_japonica.*
  Saved rpoC2_youngia_japonica.*
Fetching Taraxacum mongolicum (NC_031396.1)...
  Saved rps19_taraxacum_mongolicum.*
  Saved petD_taraxacum_mongolicum.*
  Saved psaA_taraxacum_mongolicum.*
  Saved rpl22_taraxacum_mongolicum.*
  Saved rpoC1_taraxacum_mongolicum.*
  Saved rpoC2_taraxacum_mongolicum.*
Fetching Crepis rigescens (NC_

#### Marking genes with frameshifts
Mark genes with frameshifts as putative pseudogenes using qualifier `/pseudo` and deleting qualifier `translation` in annotation.

Biopython does not support defining qualifiers without value. The `/pseudo` qualifier has no a value. So doing that manually. Copy annotation to `plastomes/table2asn/cp/02_start_codon/cp_start_codons_fixed.gb`, fixing qualifiers for genes with frameshift - petD, psaA, rpl22, rpoC1, rpoC2.
Some of them, for example rpoC1, consisys from two exons, and even might be functional if frameshift had had biological reason. To ensure if there really are frameshift but not technical artefacts, PCR and short read sequencing is necessary.

### Start codon mismatch in rps19
The sequences of the gene for both C. callicephala and C. purpurea have V at the first position.

In [27]:
# checking rps19
for gb in c_cal, c_pur:
    records = SeqIO.parse(gb, "gb")
    for record in records:
        for feature in record.features:
            if feature.type == 'CDS':
                gene = feature.qualifiers['gene'][0]
                if gene == 'rps19':
                    cds = feature.extract(record.seq)
                    aa = cds.translate(to_stop=False)
                    print(f"{cds}")

GTGACACGTTCACTAAAAAAAAATCCGTTTGTAGCGAATAATTTATTAAAAAAAATAAATAAGCTTAACACAAAAGCAGAAAAAGAAATAATAATAACCTGGTCCCGGGCATCTACCATTATACCCATAATGGTCGGCCATACTATTGCTATCCATAATGGAAAGGAACATTTACCTATTTATATAACAGATCGTATGGTAGGACACAAATTGGGAGAATTTGCACCTACTTTAAATTTCCGAGGACATGCAAAAAGCGATAATAGATCTCGTCGTTAA
GTGACACGTTCACTAAAAAAAAATCCGTTTGTAGCGAATAATTTATTAAAAAAAATAAATAAGCTTAACACAAAAGCAGAAAAAGAAATAATAATAACCTGGTCCCGGGCATCTACCATTATACCCATAATGGTCGGCCATACTATTGCTATCCATAATGGAAAGGAACATTTACCTATTTATATAACAGATCGTATGGTAGGACACAAATTGGGAGAATTTGCACCTACTTTAAATTTCCGAGGACATGCAAAAAGCGATAATAGATCTCGTCGTTAA


all _**rps19**_ genomic sequences starts from GTG but all products stored in databases starts from M. There is no special exception mark in CDS feature. A literature survey give the data that at start position TGT codes for N-formylmethionine (fMet), but in the middle of the sequence - as V.

Needs check. The initial source that was found: Sugiura et al., doi 10.1146/annurev.genet.32.1.437, which was cited by Tiller and Bock, doi 10.1093/mp/ssu022:

> Translation initiation starts with the formation of a pre-initiation complex composed of the 30S ribosomal subunit and the initiator transfer RNA (tRNA) that selects the translation initiation site in the mRNA. AUG, and in rare cases the alternative triplets GUG or UUG, serve as initiation codons in plastid mRNAs (Sugiura et al., 1998).

Manually adjusting.

### Compare genes among repeat regions

In [ ]:
df_repeats = df[df['region'].isin(['IRA', 'IRB'])]

pivot_table = df_repeats.pivot_table(
    values='feature_len',
    index='gene',
    columns=['organism', 'region'],
    aggfunc='first'
)

# Reorder levels in the columns
#pivot_table = df2.reorder_levels([1, 0], axis=1)

# Sort the columns
pivot_table = pivot_table.sort_index(axis=1)

display(pivot_table)


organism Crepis callicephala         Crepis purpurea        
region                   IRA     IRB             IRA     IRB
gene                                                        
ndhB                  1533.0  1533.0          1533.0  1533.0
ndhF                     NaN  2226.0             NaN  2226.0
rpl2                   825.0   825.0           825.0   825.0
rpl23                  282.0   282.0           282.0   282.0
rps7                   468.0   468.0           468.0   468.0
rrn16                 1490.0  1490.0          1490.0  1490.0
rrn23                 2810.0  2810.0          2810.0  2810.0
rrn4.5                 103.0   103.0           103.0   103.0
rrn5                   122.0   116.0           121.0   122.0
trnA-UGC                73.0    73.0            73.0    73.0
trnI-GAU                73.0    73.0             NaN     NaN
trnL-CAA                81.0    81.0            83.0    83.0
trnM-CAU                 NaN     NaN            75.0    75.0
trnN-GUU                72.0    72.0            74.0    74.0
trnR-ACG                74.0    74.0            75.0    75.0
trnV-GAC                73.0    72.0            72.0    72.0
ycf15                  192.0   192.0           192.0   192.0
ycf2                  6855.0  6855.0          6855.0  6855.0

#### Checking rrn5

In [30]:
# checking rps19
target = "rrn5"
for gb in [c_pur]:
    records = SeqIO.parse(gb, "gb")
    for record in records:
        for feature in record.features:
            if feature.type == 'rRNA':
                gene = feature.qualifiers['gene'][0]
                if gene == target:
                    cds = feature.extract(record.seq)
                    print(f"{cds}")

ATATTCTGGTGTCCTAGGCGTAGAGGAACCACACCAATCCATCCCGAACTTGGTGGTTAAACTCTACTGCGGTGACGATACTGTAGGGGAGGTCCTGCGGAAAAATAGCTCGACGCCAGGAT
TATTCTGGTGTCCTAGGCGTAGAGGAACCACACCAATCCATCCCGAACTTGGTGGTTAAACTCTACTGCGGTGACGATACTGTAGGGGAGGTCCTGCGGAAAAATAGCTCGACGCCAGGAT


save rRNA genes

In [32]:
import os
import time
from Bio import Entrez, SeqIO
from Bio.SeqRecord import SeqRecord

# Fill in your email address (required by NCBI Entrez)
Entrez.email = "asantaraqtasli@gmail.com"

# List of target rRNA genes (adjust to match gene names in your GenBank records)
rrna_genes = ["rrn5", "rrn5S"]

accessions = {
    "NC_007578.1": "Lactuca sativa",
    "NC_016893.1": "Lapsana communis",
    "NC_046516.1": "Youngia japonica",
    "NC_031396.1": "Taraxacum mongolicum",
    "NC_061360.1": "Crepis rigescens",
}
out_dir = "plastomes/refs"
os.makedirs(out_dir, exist_ok=True)

for acc, species in accessions.items():
    tag = species.lower().replace(" ", "_")
    print(f"Fetching {species} ({acc})...")
    
    with Entrez.efetch(db="nucleotide", id=acc, rettype="gb", retmode="text") as handle:
        rec = SeqIO.read(handle, "gb")
        
    for gene in rrna_genes:
        rrna_feat, gene_feat = None, None
        for feat in rec.features:
            g = feat.qualifiers.get("gene", [""])[0]
            if g == gene:
                if feat.type == "rRNA":
                    rrna_feat = feat
                elif feat.type == "gene":
                    gene_feat = feat
                    
        if not rrna_feat:
            print(f"  Skipping {gene}: rRNA feature not found.")
            continue
            
        # Extract sequences
        rrna_seq = rrna_feat.extract(rec.seq)
        genomic_seq = gene_feat.extract(rec.seq) if gene_feat else rrna_feat.extract(rec.seq)
        
        # Prepare output files
        gene_clean = gene.replace(" ", "_")
        prefix = f"{gene_clean}_{tag}"
        seq_map = {
            "genomic.fna": genomic_seq,
            "rrna.fna": rrna_seq
        }
        
        for ext, seq in seq_map.items():
            filepath = os.path.join(out_dir, f"{prefix}.{ext}")
            SeqIO.write(SeqRecord(seq, id=prefix), filepath, "fasta")
            
        print(f"  Saved {prefix}.*")
        time.sleep(0.5)

Fetching Lactuca sativa (NC_007578.1)...
  Skipping rrn5: rRNA feature not found.
  Skipping rrn5S: rRNA feature not found.
Fetching Lapsana communis (NC_016893.1)...
  Skipping rrn5: rRNA feature not found.
  Skipping rrn5S: rRNA feature not found.
Fetching Youngia japonica (NC_046516.1)...
  Saved rrn5_youngia_japonica.*
  Skipping rrn5S: rRNA feature not found.
Fetching Taraxacum mongolicum (NC_031396.1)...
  Skipping rrn5: rRNA feature not found.
  Skipping rrn5S: rRNA feature not found.
Fetching Crepis rigescens (NC_061360.1)...
  Saved rrn5_crepis_rigescens.*
  Skipping rrn5S: rRNA feature not found.


Among references, all _**rrn5**_ genes start with TATTCT, but not ATATTCT. The position should be adjusted. The gene located in IRb is longer, so its coordinates should be adjusted. It is in (+) strand, so the start position should be increased by 1 bp.

In [42]:
df_rRNA = df[df['type'] == 'rRNA']
df_rrn5 = df_rRNA[df_rRNA['gene'] == 'rrn5']

display(df_rrn5)

,organism,gene,start,end,gene_len,spanned,strand,region,type,feature_len,na_seq,aa_seq
101,Crepis callicephala,rrn5,104930,105046,116,False,1.0,IRB,rRNA,116.0,TGGTGT...CAGGAT,NaN
120,Crepis callicephala,rrn5,126748,126870,122,False,-1.0,IRA,rRNA,122.0,ATATTC...CAGGAT,NaN
236,Crepis purpurea,rrn5,104940,105062,122,False,1.0,IRB,rRNA,122.0,ATATTC...CAGGAT,NaN
255,Crepis purpurea,rrn5,126778,126899,121,False,-1.0,IRA,rRNA,121.0,TATTCT...CAGGAT,NaN


In [50]:
# checking rps19
target = "rrn5"
for gb in [c_pur]:
    records = SeqIO.parse(gb, "gb")
    for record in records:
        for feature in record.features:
            if feature.type == 'rRNA':
                gene = feature.qualifiers['gene'][0]
                if gene == target:
                    cds = feature.extract(record.seq)
                    print(f"{cds}")
        start = 104941
        end = 105062

        print(f"genome slice [{start}:{end}] (oldstart: 104940)\n{record.seq[start:end]}")

ATATTCTGGTGTCCTAGGCGTAGAGGAACCACACCAATCCATCCCGAACTTGGTGGTTAAACTCTACTGCGGTGACGATACTGTAGGGGAGGTCCTGCGGAAAAATAGCTCGACGCCAGGAT
TATTCTGGTGTCCTAGGCGTAGAGGAACCACACCAATCCATCCCGAACTTGGTGGTTAAACTCTACTGCGGTGACGATACTGTAGGGGAGGTCCTGCGGAAAAATAGCTCGACGCCAGGAT
genome slice [104941:105062] (oldstart: 104940)
TATTCTGGTGTCCTAGGCGTAGAGGAACCACACCAATCCATCCCGAACTTGGTGGTTAAACTCTACTGCGGTGACGATACTGTAGGGGAGGTCCTGCGGAAAAATAGCTCGACGCCAGGAT


#### Update product description in ribosomal RNA
As of organism eukariotic, but description has phrase for example "4.5S ribosomal RNA", the validation program accasionly recognize it as berlonged to Bacteria. This results in warning. The product description should contain a mark of plastid localisation: "4.5S ribosomal RNA (chloroplast)".

### tRNA fixing

In [52]:
df_tRNA = df[df['type'] == 'tRNA']
display(df_tRNA)

pivot_tRNA_len = df_tRNA.pivot_table(
    values='feature_len',
    index='gene',
    columns=['organism'],
    aggfunc='first'
)

piv = pivot_tRNA_len.sort_index(axis=1)

# Keep only rows where values differ (excluding NaN mismatches)
diff_mask = (piv['Crepis callicephala'] != piv['Crepis purpurea']) #& \
#            piv['Crepis callicephala'].notna() & \
#            piv['Crepis purpurea'].notna()

diff_genes = piv[diff_mask]
display(diff_genes)

,organism,gene,start,end,gene_len,spanned,strand,region,type,feature_len,na_seq,aa_seq
1,Crepis callicephala,trnH-GUG,2,77,75,False,-1.0,LSC,tRNA,75.0,GCGGAT...TCGCCC,NaN
3,Crepis callicephala,trnK-UUU,1758,4358,2600,False,-1.0,LSC,tRNA,74.0,TGGGTT...ACCCAT,NaN
7,Crepis callicephala,trnQ-UUG,7243,7316,73,False,-1.0,LSC,tRNA,73.0,TGGGGC...CCCAGA,NaN
10,Crepis callicephala,trnS-GCU,8489,8577,88,False,-1.0,LSC,tRNA,88.0,GGAGAG...TTTCCG,NaN
11,Crepis callicephala,trnC-GCA,9330,9402,72,False,1.0,LSC,tRNA,72.0,GGCGGC...CGCCTC,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
254,Crepis purpurea,trnR-ACG,126449,126524,75,False,-1.0,IRA,tRNA,75.0,GGGCCT...GCCCAC,NaN
258,Crepis purpurea,trnA-UGC,130306,131199,893,False,-1.0,IRA,tRNA,73.0,GGGGAT...TCTCCA,NaN
261,Crepis purpurea,trnV-GAC,134111,134183,72,False,-1.0,IRA,tRNA,72.0,AGGGAT...TCCCTA,NaN
265,Crepis purpurea,trnL-CAA,140415,140498,83,False,1.0,IRA,tRNA,83.0,TGCCTT...AGGCAT,NaN


organism,Crepis callicephala,Crepis purpurea
gene,,
trnF-GAA,70.0,74.0
trnG-GCC,71.0,72.0
trnI-GAU,73.0,NaN
trnL-CAA,81.0,83.0
trnN-GUU,72.0,74.0
trnP-UGG,75.0,76.0
trnQ-UUG,73.0,72.0
trnR-ACG,74.0,75.0
trnS-CGA,NaN,91.0


#### Run Aragorn
Use the code used for cc

In [54]:
import subprocess
import sys

gb = "plastomes/table2asn/cp/04_rrn5_coords/rrn5_fixed.gb"
aragorn_out = f"{workdir}/05_trna_aragorn/crepis_purpurea.aragorn.trnas.txt"
os.makedirs(f"{workdir}/05_trna_aragorn", exist_ok=True)

aragorn_cmd = [
    "conda", "run", "-n", "aragorn-env",
    "aragorn",
    "-v", "-e",
    "-gcbact", 
    "-c", "-i", 
    "-seq", 
    "-o", aragorn_out, 
    gb,
]

try:
    result = subprocess.run(aragorn_cmd, check=True, capture_output=True, text=True)
    print("STDOUT:", result.stdout, flush=True)
    print("STDERR:", result.stderr, flush=True)
    print("`ARAGORN` run completed successfully!", flush=True)
except subprocess.CalledProcessError as e:
    print(f"`ARAGORN` failed with exit code {e.returncode}")
    print("--- STDOUT ---")
    print(e.stdout)
    print("--- STDERR ---")
    print(e.stderr)
    sys.exit(e.returncode)

STDOUT: 
STDERR: Crepis_purpurea Crepis purpurea chloroplast.
150007 nucleotides in sequence
Mean G+C content = 37.7%
Using bacterial/plant chloroplast genetic code
Searching from 3120 to 16881
tRNA-Cys(gca) at [9349,9420] (107.847)
tRNA-Cys(gca) [9349,9420] (116.002) replacing tRNA-Cys(gca) [9349,9420] (107.847)
tRNA-Glu(ttc) at c[11938,12009] (104.658)
tRNA-Glu(ttc) c[11937,12010] (115.411) replacing tRNA-Glu(ttc) c[11938,12009] (104.658)
tRNA-Tyr(gta) at c[11751,11834] (105.503)
tRNA-Tyr(gta) c[11749,11835] (117.555) replacing tRNA-Tyr(gta) c[11751,11834] (105.503)
tRNA-Asp(gtc) at c[11564,11637] (108.811)
tRNA-Asp(gtc) c[11562,11638] (117.885) replacing tRNA-Asp(gtc) c[11564,11637] (108.811)
tRNA-Ser(gct) at c[8491,8577] (104.217)
tRNA-Ser(gct) c[8491,8578] (116.445) replacing tRNA-Ser(gct) c[8491,8577] (104.217)
tRNA-Ser(gct) c[8490,8577] (123.247) replacing tRNA-Ser(gct) c[8491,8578] (116.445)
tRNA-Gln(ttg) at c[7245,7316] (105.114)
tRNA-Gln(ttg) c[7245,7316] (105.602) replacing 

Parse intron-containing tRNAs from Aragorn output

In [56]:
import re
import pandas as pd

aragorn_report = f'{workdir}/05_trna_aragorn/crepis_purpurea.aragorn.trnas.txt'

# Regex patterns based on the parsing rules
# Rule 1: Index line (e.g., "1.")
pattern_idx = re.compile(r'^(\d+)\.$')
# Rule 5: tRNA label
pattern_label = re.compile(r'^\s+(tRNA-.*)$')
# Rule 6: Sequence length (extracts the number before "bases")
pattern_len = re.compile(r'^\s+(\d+)\s+bases')
# Rule 7: Coordinates ("c[3,77]" or "[3,77]")
pattern_coords = re.compile(r'^\s+Sequence\s+(c?\[[\d,]+\])')
# Rule 7: Intron block start marker
pattern_intron_start = re.compile(r'^Intron from ')
# Rule 9: Intron length
pattern_intron_len = re.compile(r'^Intron Length:\s+(\d+)')
# Rule 10: Intron position and surrounding sequence
pattern_intron_pos = re.compile(r'^Intron Insertion Position\(([\d-]+)\):\s*(.*)$')

records = []
current_record = {}
in_intron_block = False

with open(aragorn_report, 'r') as f:
    for line in f:
        # Check for new tRNA record start
        match_idx = pattern_idx.match(line)
        if match_idx:
            # If we have a current record, save it before starting a new one
            if current_record:
                records.append(current_record)
            
            # Initialize new record with default values
            current_record = {
                'idx': match_idx.group(1),
                'tRNA_label': None,
                'sequence_length': None,
                'coordinates': None,
                'intron_presence': 0,
                'intron_length': None,
                'intron_start': None,
                'intron_surrounding': None
            }
            in_intron_block = False
            continue

        # Skip header lines and footer lines that don't belong to a record
        if not current_record:
            continue

        # Parse tRNA Description (Rules 4-7)
        # We parse sequentially to ensure we grab the correct lines
        if not in_intron_block:
            # Rule 5: tRNA label
            if not current_record['tRNA_label']:
                match = pattern_label.match(line)
                if match:
                    current_record['tRNA_label'] = match.group(1).strip()
                    continue
            
            # Rule 6: Length
            if current_record['tRNA_label'] and not current_record['sequence_length']:
                match = pattern_len.match(line)
                if match:
                    current_record['sequence_length'] = int(match.group(1))
                    continue

            # Rule 7: Coordinates
            if current_record['sequence_length'] and not current_record['coordinates']:
                match = pattern_coords.match(line)
                if match:
                    current_record['coordinates'] = match.group(1).strip()
                    continue

        # Parse Intron Data (Rules 7-11)
        # Check for Intron block header
        if pattern_intron_start.match(line):
            in_intron_block = True
            current_record['intron_presence'] = 1
            continue

        if in_intron_block:
            # Rule 9: Intron Length
            if not current_record['intron_length']:
                match = pattern_intron_len.match(line)
                if match:
                    current_record['intron_length'] = int(match.group(1))
                    continue
            
            # Rule 10: Intron Position and Surrounding
            if not current_record['intron_start']:
                match = pattern_intron_pos.match(line)
                if match:
                    current_record['intron_start'] = match.group(1)
                    current_record['intron_surrounding'] = match.group(2).strip()
                    # Continue parsing (to consume any remaining intron sequence lines)
                    continue

# Append the last record after EOF
if current_record:
    records.append(current_record)

# Create DataFrame
df_aragorn_raw = pd.DataFrame(records)
df_aragorn_raw['intron_length'] = df_aragorn_raw['intron_length'].astype('Int64')

# Ensure column order matches requirements
columns = [
    'idx', 'tRNA_label', 'sequence_length', 'coordinates', 
    'intron_presence', 'intron_length', 'intron_start', 'intron_surrounding'
]

df_intron = df_aragorn_raw[df_aragorn_raw['intron_presence'] == 1]

# df_intron was created via filtering, so it is a view
# Forse copying to avoid SettingWithCopyWarning exception
if df_intron._is_copy:
    df_intron = df_intron.copy()


display(df_intron)

,idx,tRNA_label,sequence_length,coordinates,intron_presence,intron_length,intron_start,intron_surrounding
1,2,tRNA-Lys(ttt),74,"c[1761,4360]",1,2526,38-39,tttaa-Intron-ccgac
9,10,tRNA-Ser(cga),91,"c[30130,30911]",1,691,31-32,tgatt-Intron-tcgac
15,16,tRNA-Leu(taa),85,"[46805,47329]",1,440,35-36,gactt-Intron-aaaat
17,18,tRNA-Val(tac),73,"c[51120,51762]",1,570,38-39,tacac-Intron-cgaga
24,25,tRNA-Glu(ttc),73,"[99742,100577]",1,763,33-34,cccct-Intron-ttcac
25,26,tRNA-Ala(tgc),73,"[100642,101534]",1,820,37-38,ttgca-Intron-cggcg
31,32,tRNA-Ala(tgc),73,"c[130307,131199]",1,820,37-38,ttgca-Intron-cggcg
32,33,tRNA-Glu(ttc),73,"c[131264,132099]",1,763,33-34,cccct-Intron-ttcac


calculate exon exact positions

In [57]:
# Prepare parser
def calc_joins(coordinates: str, 
               intron_length: int,
               intron_start: str,
               sequence_length: int,
               ) -> dict:
    exact_start = int(intron_start.split("-")[0])
    
    location = {}
    if 'c[' in coordinates:
        location['strand'] = -1
    else:
        location['strand'] = 1

    coords_list = coordinates.strip('c[]').split(",")
    coords_list = list(map(int, coords_list))
    coords_list.sort()
    location['start'] = coords_list[0]
    location['end'] = coords_list[1]

    # exon1: first `exact_start` bases from genomic start
    exon1_loc = [location['start'], location['start'] + exact_start - 1]
    # exon2: remaining bases anchored to genomic end
    exon2_bases = sequence_length - exact_start
    exon2_loc = [location['end'] - exon2_bases + 1, location['end']]

    # validation (pure arithmetic, no +1 fudge needed)
    calculated_l = (exon1_loc[1] - exon1_loc[0] + 1) + (exon2_loc[1] - exon2_loc[0] + 1)
    if sequence_length == calculated_l:
        print("lengths match!")
    else:
        print(f"lengths do not match: {calculated_l} and expected {sequence_length}")

    return [location['strand'], [exon1_loc, exon2_loc]]


df_intron[['strand', 'joined_coordinates']] = df_intron.apply(
    lambda row: calc_joins(
        coordinates=row['coordinates'], 
        intron_length=row['intron_length'], 
        intron_start=row['intron_start'],
        sequence_length=row['sequence_length']
    ), 
    axis=1,
    result_type='expand'
)

display(df_intron)

lengths match!
lengths match!
lengths match!
lengths match!
lengths match!
lengths match!
lengths match!
lengths match!


,idx,tRNA_label,sequence_length,coordinates,intron_presence,intron_length,intron_start,intron_surrounding,strand,joined_coordinates
1,2,tRNA-Lys(ttt),74,"c[1761,4360]",1,2526,38-39,tttaa-Intron-ccgac,-1,"[[1761, 1798], [4325, 4360]]"
9,10,tRNA-Ser(cga),91,"c[30130,30911]",1,691,31-32,tgatt-Intron-tcgac,-1,"[[30130, 30160], [30852, 30911]]"
15,16,tRNA-Leu(taa),85,"[46805,47329]",1,440,35-36,gactt-Intron-aaaat,1,"[[46805, 46839], [47280, 47329]]"
17,18,tRNA-Val(tac),73,"c[51120,51762]",1,570,38-39,tacac-Intron-cgaga,-1,"[[51120, 51157], [51728, 51762]]"
24,25,tRNA-Glu(ttc),73,"[99742,100577]",1,763,33-34,cccct-Intron-ttcac,1,"[[99742, 99774], [100538, 100577]]"
25,26,tRNA-Ala(tgc),73,"[100642,101534]",1,820,37-38,ttgca-Intron-cggcg,1,"[[100642, 100678], [101499, 101534]]"
31,32,tRNA-Ala(tgc),73,"c[130307,131199]",1,820,37-38,ttgca-Intron-cggcg,-1,"[[130307, 130343], [131164, 131199]]"
32,33,tRNA-Glu(ttc),73,"c[131264,132099]",1,763,33-34,cccct-Intron-ttcac,-1,"[[131264, 131296], [132060, 132099]]"


Manually adjust splaced tRNA coordinates in `plastomes/table2asn/cp/05_trna_aragorn/trna_fixed.gb` file. The coords reported by Aragorn seems to already be in GenBank coordinates format. Duplicated intron features were identified for absent trnG-UCC (c\[30179..30888\]) in place of trnS-CGA intron. Gene name was changed to trnE-UUC (c\[131264//132099\])

#### Parse total Aragorn output

In [58]:
# parse Aragorn output
import pandas as pd

aragorn_out = f"{workdir}/05_trna_aragorn/crepis_purpurea.aragorn.trnas.txt"
list_parsed = []

with open(aragorn_out, "r") as handle:
    start_line = None
    for i, line in enumerate(handle):
        line_parts = []
        if "Aragorn" in line:
            if "GenBank" in line and "Aragorn" in line:
                line_parts = line.strip(" ").split()
                print(line_parts)
                if len(line_parts) == 2:
                    start_line = i
                    print(f"start line index: {start_line}")
        if start_line:
            if i == start_line:
                pass
            else:
                gb_label = None
                gb_coord = None
                arag_note = None
                arag_label = None
                arag_coord = None
                arag_score = None
                line = line.split("\n")[0]
                #print(i, line)
                inconsistent = False
                split_initial = line.split(" ", maxsplit=1)
                consistency_flag = split_initial[0]
                if len(split_initial) < 2:
                    break
                else:
                    remain_part = line.split(" ", maxsplit=1)[1].strip()
                    line_parts.append(consistency_flag)
                    line_parts.extend(remain_part.split(maxsplit=5))
                    #print(line_parts)
                    if "*" in line_parts[0]:
                        inconsistent = True
                    gb_label = line_parts[1]
                    gb_coord = line_parts[2]
                    if 'Not' in line_parts[3]:
                        pass
                    else:
                        arag_label = line_parts[3]
                        arag_coord = line_parts[4]
                        arag_score = line_parts[5]
                        #print(f"'line_parts' length: {len(line_parts)}")
                        if len(line_parts) > 6:
                            arag_note = line_parts[-1]
                    #print(f"{inconsistent}\t{gb_label}\t{gb_coord}\t{arag_label}\t{arag_coord}\t{arag_score}\t{arag_note}")
                    line_dict = {
                        'inconsistent': inconsistent,
                        'gb_label': gb_label,
                        'gb_coord': gb_coord,
                        'arag_label': arag_label,
                        'arag_coord': arag_coord,
                        'arag_score': arag_score,
                        'arag_note': arag_note
                    }

                    list_parsed.append(line_dict)
                    #print(line_dict)

df_aragorn = pd.DataFrame.from_dict(list_parsed)
display(df_aragorn)

['GenBank', 'to', 'Aragorn', 'comparison']
['GenBank', 'Aragorn']
start line index: 2061


,inconsistent,gb_label,gb_coord,arag_label,arag_coord,arag_score,arag_note
0,False,tRNA-His,"c(6,80)",tRNA-His(gtg),"c[6,80]",115.233,None
1,True,tRNA-???,"c(1761,1796)",tRNA-Lys(ttt),"c[1761,4360]",103.307,amino acceptor and sequence length mismatch
2,False,tRNA-Gln,"c(7245,7316)",tRNA-Gln(ttg),"c[7245,7316]",116.786,None
3,True,tRNA-???,"c(8490,8577)",tRNA-Ser(gct),"c[8490,8577]",123.247,amino acceptor mismatch
4,True,tRNA-???,"(9349,9420)",tRNA-Cys(gca),"[9349,9420]",116.002,amino acceptor mismatch
5,False,tRNA-Asp,"c(11562,11638)",tRNA-Asp(gtc),"c[11562,11638]",117.885,None
6,False,tRNA-Tyr,"c(11749,11835)",tRNA-Tyr(gta),"c[11749,11835]",117.555,None
7,False,tRNA-Glu,"c(11937,12010)",tRNA-Glu(ttc),"c[11937,12010]",115.411,None
8,True,tRNA-???,"c(29854,29925)",tRNA-Arg(tct),"c[29854,29925]",117.452,amino acceptor mismatch
9,True,tRNA-???,"c(30130,30189)",tRNA-Ser(cga),"c[30130,30911]",101.783,amino acceptor and sequence length mismatch


#### Validate changes with Aragorn

In [59]:
import subprocess
import sys

gb = f"{workdir}/05_trna_aragorn/trna_fixed.gb"
aragorn_out = f"{workdir}/05_trna_aragorn/crepis_purpurea.aragorn.trnas.validation.txt"
os.makedirs(f"{workdir}/05_trna_aragorn", exist_ok=True)

aragorn_cmd = [
    "conda", "run", "-n", "aragorn-env",
    "aragorn",
    "-v", "-e",
    "-gcbact", 
    "-c", "-i", 
    "-seq", 
    "-o", aragorn_out, 
    gb,
]

try:
    result = subprocess.run(aragorn_cmd, check=True, capture_output=True, text=True)
    print("STDOUT:", result.stdout, flush=True)
    print("STDERR:", result.stderr, flush=True)
    print("`ARAGORN` run completed successfully!", flush=True)
except subprocess.CalledProcessError as e:
    print(f"`ARAGORN` failed with exit code {e.returncode}")
    print("--- STDOUT ---")
    print(e.stdout)
    print("--- STDERR ---")
    print(e.stderr)
    sys.exit(e.returncode)

STDOUT: 
STDERR: Crepis_purpurea Crepis purpurea chloroplast.
150007 nucleotides in sequence
Mean G+C content = 37.7%
Using bacterial/plant chloroplast genetic code
Searching from 3120 to 16881
tRNA-Cys(gca) at [9349,9420] (107.847)
tRNA-Cys(gca) [9349,9420] (116.002) replacing tRNA-Cys(gca) [9349,9420] (107.847)
tRNA-Glu(ttc) at c[11938,12009] (104.658)
tRNA-Glu(ttc) c[11937,12010] (115.411) replacing tRNA-Glu(ttc) c[11938,12009] (104.658)
tRNA-Tyr(gta) at c[11751,11834] (105.503)
tRNA-Tyr(gta) c[11749,11835] (117.555) replacing tRNA-Tyr(gta) c[11751,11834] (105.503)
tRNA-Asp(gtc) at c[11564,11637] (108.811)
tRNA-Asp(gtc) c[11562,11638] (117.885) replacing tRNA-Asp(gtc) c[11564,11637] (108.811)
tRNA-Ser(gct) at c[8491,8577] (104.217)
tRNA-Ser(gct) c[8491,8578] (116.445) replacing tRNA-Ser(gct) c[8491,8577] (104.217)
tRNA-Ser(gct) c[8490,8577] (123.247) replacing tRNA-Ser(gct) c[8491,8578] (116.445)
tRNA-Gln(ttg) at c[7245,7316] (105.114)
tRNA-Gln(ttg) c[7245,7316] (105.602) replacing 

### Checking with GB2Sequin

Error are still identified:
WARNING: valid [SEQ_FEAT.TranslExcept] Suspicious transl_except X at first codon of complete CDS FEATURE: CDS: photosystem II subunit L [lcl|Crepis_purpurea:c62025-61909] [lcl|Crepis_purpurea: raw, dna len= 150007] -> [lcl|Crepis_purpurea_32]
ERROR: valid [SEQ_FEAT.StartCodon] Ambiguous start codon used. Wrong genetic code [11] or protein should be partial FEATURE: CDS: photosystem II subunit L [lcl|Crepis_purpurea:c62025-61909] [lcl|Crepis_purpurea: raw, dna len= 150007] -> [lcl|Crepis_purpurea_32]
WARNING: valid [SEQ_FEAT.TranslExcept] Suspicious transl_except X at first codon of complete CDS FEATURE: CDS: NADH dehydrogenase subunit D [lcl|Crepis_purpurea:c113924-112422] [lcl|Crepis_purpurea: raw, dna len= 150007] -> [lcl|Crepis_purpurea_67]
ERROR: valid [SEQ_FEAT.StartCodon] Ambiguous start codon used. Wrong genetic code [11] or protein should be partial FEATURE: CDS: NADH dehydrogenase subunit D [lcl|Crepis_purpurea:c113924-112422] [lcl|Crepis_purpurea: raw, dna len= 150007] -> [lcl|Crepis_purpurea_67]


But exceptions for these issues are listed. Looking for table2asn output.

In [60]:
# run table2asn
import subprocess
import os
import sys

table2asn = "bin/table2asn.linux64"

input_dir = f"{workdir}/07_val_table2asn"
out_dir = input_dir
template = f"{input_dir}/template.sbt"

os.makedirs(out_dir, exist_ok=True)

cmd = [
    table2asn,
    "-indir", str(os.path.abspath(input_dir)),
    "-t", str(os.path.abspath(template)),
    "-V", "vb",
    "-verbose",
    "-Z",
    "-o", os.path.join(out_dir, "output"),
]

try:
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)
    print("table2asn run completed")
except subprocess.CalledProcessError as e:
    print("ERROR:", e.stderr)
    sys.exit(e.returncode)

ERROR: Will be using one threads
Recognized annotation format: five-column feature table
Problem:        Unrecognized qualifier name
SeqId:          lcl|Crepis_purpurea
Line:           351
FeatureName:    gene
QualifierName:  transl_table
QualifierValue: 11


Falling back on built-in data for popular organisms.



SystemExit: 2

/home/asan/miniconda3/envs/plastome_post-env/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


Errors are still identified. Looking for output with cc genome.

In [ ]:
# run table2asn
import subprocess
import os
import sys

table2asn = "bin/table2asn.linux64"

input_dir = f"{workdir}/08_val_cc_table2asn_DONT_USE"
out_dir = input_dir
template = f"{input_dir}/template.sbt"

os.makedirs(out_dir, exist_ok=True)

cmd = [
    table2asn,
    "-indir", str(os.path.abspath(input_dir)),
    "-t", str(os.path.abspath(template)),
    "-V", "vb",
    "-verbose",
    "-Z",
    "-o", os.path.join(out_dir, "output"),
]

try:
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)
    print("table2asn run completed")
except subprocess.CalledProcessError as e:
    print("ERROR:", e.stderr)
    sys.exit(e.returncode)

STDOUT:
 
STDERR:
 Will be using one threads
Recognized annotation format: five-column feature table

table2asn run completed


For cc, no errors reported. In gb files, inputted into the GB2Sequin, the features related to genes with start codon errors were annotated the same way. In the tab-formatted annotations, the issue was identified:

In cc, the exception line had ":" in aa defining expression, but in cp, the "=" sine was used.
cc  transl_except	(pos:complement(61995..61997),aa:Met)
cp  transl_except	(pos:complement(62023..62025),aa=M)

fixing the tbl file and trying again to validate via `table2asn`. Also, "trans_table" qualifier line was removed in line 351 for accD gene because genes doesn't have such qualifier, but only CDS have.

In [63]:
# run table2asn
import subprocess
import os
import sys

table2asn = "bin/table2asn.linux64"

input_dir = f"{workdir}/07_val_table2asn"
out_dir = input_dir
template = f"{input_dir}/template.sbt"

os.makedirs(out_dir, exist_ok=True)

cmd = [
    table2asn,
    "-indir", str(os.path.abspath(input_dir)),
    "-t", str(os.path.abspath(template)),
    "-V", "vb",
    "-verbose",
    "-Z",
    "-o", os.path.join(out_dir, "output"),
]

try:
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)
    print("table2asn run completed")
except subprocess.CalledProcessError as e:
    print("ERROR:", e.stderr)
    sys.exit(e.returncode)

STDOUT:
 
STDERR:
 Will be using one threads
Recognized annotation format: five-column feature table
Falling back on built-in data for popular organisms.

table2asn run completed


Validation was performed well. Also adding source line and updating header for Crepis purpurea tbl file.

the frameshift errors related to 6 genes whose CDS were removed needs fixing. NCBI does not accept the data.